In [1]:
# import libraries
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [2]:
class BatsmanState(TypedDict):

    runs:int
    balls:int
    fours:int
    sixes:int

    sr: float
    bpb: float
    boundary_percent: float
    summary: str

In [3]:
def calculate_sr(state: BatsmanState):
    sr = (state['runs']/state['balls'])*100

    return {'sr': sr}

In [4]:
def calculate_bpb(state: BatsmanState):
    bpb = state['balls'] / (state['fours'] + state['sixes'])

    return {'bpb' : bpb}

In [5]:
def calculate_boundary_percent(state: BatsmanState):
    boundary_percent = 100*(4*state['fours'] + 6*state['sixes'])/state['runs']

    return {'boundary_percent' : boundary_percent}

In [6]:
def summary(state: BatsmanState):
    summary = f"""
Strike Rate : {state['sr']}
Balls per boundary : {state['bpb']}
Boundary percent : {state['boundary_percent']}
"""
    return {'summary' : summary}

In [7]:
graph = StateGraph(BatsmanState)

# add nodes to graph
graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)

# add egdes to graph
graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percent')

graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)


# compile graph
workflow = graph.compile()

In [9]:
initial_state = {
    'runs' : 40,
    'balls' : 10,
    'fours' : 2,
    'sixes' : 4
}

final_state = workflow.invoke(initial_state)

final_state

{'runs': 40,
 'balls': 10,
 'fours': 2,
 'sixes': 4,
 'sr': 400.0,
 'bpb': 1.6666666666666667,
 'boundary_percent': 80.0,
 'summary': '\nStrike Rate : 400.0\nBalls per boundary : 1.6666666666666667\nBoundary percent : 80.0\n'}

In [10]:
print(final_state['summary'])


Strike Rate : 400.0
Balls per boundary : 1.6666666666666667
Boundary percent : 80.0

